# CREAM Noisy-Graph Analysis

Standalone CREAM trained on **one expert graph** — the baseline for mCREAM ensemble comparison.

**Experiment:** `simple_main.py` with `cream_noisy_ensemble` configs.
Each run uses one expert graph from the same set used in the ensemble experiments.

**Purpose:** Show that standalone CREAM on one noisy graph achieves ~90% (close to baseline),
while mCREAM ensemble of 5 noisy graphs achieves higher — the key comparison.

**Noise families:** deletion / addition / reversal × low / medium / high
**Experts:** 0..4 (same graphs as ensemble experiments)
**Datasets:** Complete_Concept_FMNIST, CelebA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# ── CONFIGURE ────────────────────────────────────────────────────────────────
EXPERIMENTS_ROOT = Path('/home/dani00003/mCREAM/experiments')
DAG_CFMNIST      = Path('/home/dani00003/mCREAM/data/FashionMNIST/Complete_Concept_FMNIST_DAG.csv')
DAG_CELEBA       = Path('/home/dani00003/mCREAM/data/CelebA/final_DAG_unfair.csv')

DATASETS = ['Complete_Concept_FMNIST', 'CelebA']
ACTIONS  = ['deletion', 'addition', 'reversal']
LEVELS   = ['low', 'medium', 'high']
N_EXPERTS = 5

ACTION_COLOR = {'deletion': '#e74c3c', 'addition': '#2ecc71', 'reversal': '#3498db'}

print(f'Experiments root exists: {EXPERIMENTS_ROOT.exists()}')

## 1. Load Results

In [ ]:
def load_cream_noisy_results(root: Path) -> pd.DataFrame:
    """
    Load results from cream_noisy_ensemble experiments.
    simple_main.py saves CSVs to:
      experiments/{dataset}/train_cbm/{model}/{config_folder}/last_metrics/{exp_name}.csv
    where config_folder = cream_noisy_ensemble
    """
    rows = []
    model_map = {
        'Complete_Concept_FMNIST': 'Standard_FashionMNIST',
        'CelebA': 'Standard_CelebA',
    }
    for dataset in DATASETS:
        model_name = model_map.get(dataset, 'Standard_FashionMNIST')
        base_dir = root / dataset / 'train_cbm' / model_name / 'cream_noisy_ensemble'
        if not base_dir.exists():
            print(f'  [SKIP] {base_dir}')
            continue

        # CSVs saved to last_metrics/
        metrics_dir = base_dir / 'last_metrics'
        if not metrics_dir.exists():
            print(f'  [SKIP] no last_metrics in {base_dir}')
            continue

        for csv_f in sorted(metrics_dir.glob('cream_noisy_*.csv')):
            name = csv_f.stem  # cream_noisy_deletion_medium_expert0
            parts = name.split('_')
            try:
                action = parts[2]
                level  = parts[3]
                expert = int(parts[4].replace('expert', ''))
            except (IndexError, ValueError):
                continue
            try:
                df = pd.read_csv(csv_f)
                if len(df) == 0: continue
                df['dataset']     = dataset
                df['exp_name']    = name
                df['action']      = action
                df['noise_level'] = level
                df['expert']      = expert
                rows.append(df)
            except Exception as e:
                print(f'  ERR {csv_f}: {e}')

    if not rows:
        print('No cream_noisy_ensemble results found.')
        print('Run submit_cream_noisy_ensemble.sh and wait for jobs to complete.')
        return pd.DataFrame()

    df = pd.concat(rows, ignore_index=True)
    print(f'Loaded {len(df)} rows')
    print(f'  datasets:    {sorted(df["dataset"].unique())}')
    print(f'  actions:     {sorted(df["action"].unique())}')
    print(f'  noise_level: {sorted(df["noise_level"].unique())}')
    print(f'  experts:     {sorted(df["expert"].unique())}')
    return df

noisy_df = load_cream_noisy_results(EXPERIMENTS_ROOT)
noisy_df.head(3) if len(noisy_df) > 0 else None

## 2. Summary Table — Task Accuracy, CCI, PFI

In [ ]:
if len(noisy_df) == 0:
    print('No data yet.')
else:
    KEY_METRICS = [
        'test_task_accuracy', 'test_concept_accuracy',
        'CCI', 'PFI_concept_importance', 'PFI_side_importance',
        'c2y_baseline_accuracy', 'concept_leakage',
    ]
    available = [c for c in KEY_METRICS if c in noisy_df.columns]
    GROUP = ['dataset', 'action', 'noise_level', 'expert']

    means  = noisy_df.groupby(GROUP)[available].mean()
    stds   = noisy_df.groupby(GROUP)[available].std()
    counts = noisy_df.groupby(GROUP)[available[0]].count().rename('n_seeds')

    summary = pd.DataFrame(index=means.index)
    summary['n_seeds'] = counts
    for col in available:
        summary[col] = (means[col].map('{:.4f}'.format)
                        + ' ± ' + stds[col].map('{:.4f}'.format))

    for ds in DATASETS:
        if ds not in means.index.get_level_values('dataset'): continue
        sub = summary.xs(ds, level='dataset')
        print(f"\n{'─'*60}\n  {ds}\n{'─'*60}")
        display(sub)

## 3. Per-Expert Accuracy: Task & Concept

In [ ]:
if len(noisy_df) == 0:
    print('No data yet.')
else:
    for dataset in DATASETS:
        ds = noisy_df[noisy_df['dataset'] == dataset]
        if ds.empty: continue

        for metric, ylabel in [('test_task_accuracy', 'Task Accuracy'),
                                ('test_concept_accuracy', 'Concept Accuracy')]:
            if metric not in ds.columns: continue

            fig, axes = plt.subplots(1, len(ACTIONS),
                                     figsize=(5*len(ACTIONS), 5), sharey=True)
            fig.suptitle(f'{dataset} — Standalone CREAM per expert\n{ylabel}',
                         fontsize=12, fontweight='bold')

            for ax, action in zip(axes, ACTIONS):
                act = ds[ds['action'] == action]
                if act.empty: ax.set_title(f'{action} — no data'); continue

                agg = act.groupby(['noise_level', 'expert'])[metric].agg(
                    ['mean', 'std']).reset_index()

                for level in LEVELS:
                    lv = agg[agg['noise_level'] == level]
                    if lv.empty: continue
                    color = plt.cm.RdYlGn(
                        LEVELS.index(level) / (len(LEVELS) - 1))
                    ax.errorbar(lv['expert'], lv['mean'], yerr=lv['std'],
                                label=level, color=color,
                                marker='o', markersize=7, linewidth=2,
                                capsize=3)

                ax.set_title(action)
                ax.set_xlabel('Expert index')
                ax.set_ylabel(ylabel if action == ACTIONS[0] else '')
                ax.set_xticks(range(N_EXPERTS))
                ax.legend(title='Noise level', fontsize=8)

            plt.tight_layout()
            plt.show()

## 4. Robustness: Accuracy vs Noise Level (H1)

In [ ]:
if len(noisy_df) == 0:
    print('No data yet.')
else:
    level_num = {'low': 0.25, 'medium': 0.50, 'high': 0.75}

    for dataset in DATASETS:
        ds = noisy_df[noisy_df['dataset'] == dataset].copy()
        if ds.empty: continue
        ds['noise_prob'] = ds['noise_level'].map(level_num)

        # Average across experts and seeds
        agg = ds.groupby(['action', 'noise_prob'])['test_task_accuracy'].agg(
            ['mean', 'std']).reset_index()

        fig, ax = plt.subplots(figsize=(8, 5))
        fig.suptitle(f'{dataset}\nStandalone CREAM: Task Accuracy vs Noise Level',
                     fontsize=12, fontweight='bold')

        for action in ACTIONS:
            sub = agg[agg['action'] == action]
            if sub.empty: continue
            color = ACTION_COLOR.get(action, 'gray')
            ax.plot(sub['noise_prob'], sub['mean'],
                    color=color, marker='o', markersize=8,
                    linewidth=2, label=action)
            ax.fill_between(sub['noise_prob'],
                            sub['mean'] - sub['std'],
                            sub['mean'] + sub['std'],
                            alpha=0.15, color=color)

        ax.set_xlabel('Noise probability')
        ax.set_ylabel('Task Accuracy')
        ax.set_xticks([0.25, 0.50, 0.75])
        ax.set_xticklabels(['low\n(0.25)', 'medium\n(0.50)', 'high\n(0.75)'])
        ax.legend(title='Noise type', fontsize=9)
        plt.tight_layout()
        plt.savefig(f'cream_noisy_robustness_{dataset}.png', dpi=150, bbox_inches='tight')
        plt.show()

## 5. Intervention Curves

In [ ]:
def load_cream_noisy_interventions(root: Path) -> pd.DataFrame:
    rows = []
    model_map = {
        'Complete_Concept_FMNIST': 'Standard_FashionMNIST',
        'CelebA': 'Standard_CelebA',
    }
    for dataset in DATASETS:
        model_name = model_map.get(dataset, 'Standard_FashionMNIST')
        base_dir = root / dataset / 'train_cbm' / model_name / 'cream_noisy_ensemble'
        if not base_dir.exists(): continue
        for csv_f in base_dir.rglob('intervention_results.csv'):
            try:
                df = pd.read_csv(csv_f)
                # Path: .../cream_noisy_deletion_medium_expert0/lightning_logs/.../intervention_results.csv
                parts = csv_f.parts
                exp_name = None
                for i, p in enumerate(parts):
                    if p == 'cream_noisy_ensemble' and i+1 < len(parts):
                        exp_name = parts[i+1]
                if exp_name is None or not exp_name.startswith('cream_noisy'): continue
                p = exp_name.split('_')
                try:
                    action = p[2]; level = p[3]; expert = int(p[4].replace('expert',''))
                except (IndexError, ValueError): continue
                df['dataset'] = dataset; df['action'] = action
                df['noise_level'] = level; df['expert'] = expert
                rows.append(df)
            except Exception as e:
                print(f'ERR {csv_f}: {e}')
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

interv_df = load_cream_noisy_interventions(EXPERIMENTS_ROOT)
print(f'Loaded {len(interv_df)} intervention rows')

In [ ]:
if len(interv_df) == 0:
    print('No intervention data yet.')
else:
    for dataset in DATASETS:
        for action in ACTIONS:
            sub = interv_df[
                (interv_df['dataset'] == dataset) &
                (interv_df['action']  == action)
            ]
            if sub.empty: continue

            fig, axes = plt.subplots(1, len(LEVELS),
                                     figsize=(5*len(LEVELS), 5), sharey=True)
            fig.suptitle(f'{dataset} — {action} noise\n'
                         f'Standalone CREAM intervention curves (one line per expert)',
                         fontsize=12, fontweight='bold')

            for ax, level in zip(axes, LEVELS):
                lv = sub[sub['noise_level'] == level]
                if lv.empty:
                    ax.set_title(f'{level} — no data'); continue

                colors = plt.cm.tab10(np.linspace(0, 0.9, N_EXPERTS))
                for expert in range(N_EXPERTS):
                    exp_m = lv[lv['expert'] == expert]
                    if exp_m.empty: continue
                    # Use 'simple' mode if column exists, else just num_interventions
                    if 'mode' in exp_m.columns:
                        exp_m = exp_m[exp_m['mode'] == 'simple']
                    agg = (exp_m.groupby('num_interventions')['test_task_accuracy']
                               .agg(['mean', 'std']).reset_index())
                    ax.plot(agg['num_interventions'], agg['mean'],
                            color=colors[expert], linewidth=1.8,
                            marker='.', markersize=5,
                            label=f'Expert {expert}')
                    ax.fill_between(agg['num_interventions'],
                                   agg['mean'] - agg['std'],
                                   agg['mean'] + agg['std'],
                                   alpha=0.1, color=colors[expert])

                # Mean across all experts
                mean_all = (lv.groupby('num_interventions')['test_task_accuracy']
                              .agg(['mean', 'std']).reset_index())
                ax.plot(mean_all['num_interventions'], mean_all['mean'],
                        color='black', linewidth=2.5, linestyle='--',
                        marker='D', markersize=5, label='Mean (all experts)')

                ax.axhline(1.0, color='gray', linestyle=':', alpha=0.4)
                ax.set_title(f'{level} noise')
                ax.set_xlabel('Number of interventions')
                ax.set_ylabel('Task Accuracy' if level == LEVELS[0] else '')
                ax.legend(fontsize=6, loc='lower right')

            plt.tight_layout()
            plt.savefig(f'cream_noisy_interventions_{dataset}_{action}.png',
                        dpi=150, bbox_inches='tight')
            plt.show()

## 6. CCI and PFI Summary

In [ ]:
if len(noisy_df) == 0:
    print('No data yet.')
else:
    cci_cols = [c for c in ['CCI', 'PFI_concept_importance', 'PFI_side_importance']
                if c in noisy_df.columns]
    if not cci_cols:
        print('CCI/PFI columns not found yet.')
    else:
        for dataset in DATASETS:
            ds = noisy_df[noisy_df['dataset'] == dataset].dropna(subset=cci_cols[:1])
            if ds.empty: continue

            fig, axes = plt.subplots(1, len(cci_cols),
                                     figsize=(5*len(cci_cols), 5))
            if len(cci_cols) == 1: axes = [axes]
            fig.suptitle(f'Concept Importance — {dataset}\n'
                         f'Standalone CREAM on noisy graphs',
                         fontsize=12, fontweight='bold')

            for ax, metric in zip(axes, cci_cols):
                sub = ds.dropna(subset=[metric])
                if sub.empty: continue
                sns.boxplot(data=sub, x='noise_level', y=metric,
                            hue='action', order=LEVELS, ax=ax,
                            palette=ACTION_COLOR)
                ax.set_title(metric)
                ax.set_xlabel('Noise level')
                if metric == 'CCI':
                    ax.axhline(0.5, color='red', linestyle='--',
                               alpha=0.5, label='threshold=0.5')
                    ax.legend(fontsize=7)

            plt.tight_layout()
            plt.show()

## 7. Export Summary Table

In [ ]:
if len(noisy_df) > 0:
    key = ['test_task_accuracy', 'test_concept_accuracy',
           'CCI', 'PFI_concept_importance', 'intervention_acc_max']
    available = [c for c in key if c in noisy_df.columns]
    agg = (noisy_df.groupby(['dataset', 'action', 'noise_level', 'expert'])[available]
                   .agg(['mean', 'std']).round(4))
    agg.to_csv('cream_noisy_ensemble_summary.csv')
    print('Saved: cream_noisy_ensemble_summary.csv')
    display(agg)